# Nível 1 — Dados e primeira análise com LLM

**Desafio Técnico — Estágio em Engenharia de Inteligência Artificial**

Objetivo: tratar a base, normalizar valores para BRL, produzir agregações, implementar e validar as regras determinísticas e realizar análise estruturada com LLM.

> **Princípio de arquitetura:** pandas calcula; a LLM interpreta. Soma, mediana, contagem e comparação com limites não são delegadas à LLM.

## 1. Configuração, imports e carregamento

O notebook localiza automaticamente a raiz do repositório. No Google Colab, a chave é lida de `Secrets`; localmente, pode vir de `.env`. Nenhuma credencial é gravada no notebook.

In [22]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

candidatos = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/desafio-estagio-engenharia-ia"),
]
BASE_DIR = next(
    (p for p in candidatos if (p / "dados" / "dados_nivel_1.json").exists()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError(
        "Não encontrei dados/dados_nivel_1.json. "
        "Execute o notebook a partir do repositório do projeto."
    )

load_dotenv(BASE_DIR / ".env")

try:
    from google.colab import userdata
    chave_colab = userdata.get("GEMINI_API_KEY")
    if chave_colab:
        os.environ["GEMINI_API_KEY"] = chave_colab
except Exception:
    pass

os.environ.setdefault("GEMINI_MODEL", "gemini-3.5-flash-lite")

DADOS_PATH = BASE_DIR / "dados" / "dados_nivel_1.json"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

with open(DADOS_PATH, encoding="utf-8") as f:
    payload = json.load(f)

taxa_cambio = float(payload["taxa_cambio_usd_brl"])
df_raw = pd.DataFrame(payload["operacoes"])

print("Raiz do projeto:", BASE_DIR)
print(f"Taxa USD/BRL fornecida: {taxa_cambio}")
print(f"Registros recebidos: {len(df_raw)}")
print(f"Clientes distintos: {df_raw['cliente_id'].nunique()}")
print("Gemini configurado:", bool(os.getenv("GEMINI_API_KEY", "").strip()))
print("Modelo:", os.environ["GEMINI_MODEL"])
display(df_raw.head())

Raiz do projeto: /content/desafio-estagio-engenharia-ia
Taxa USD/BRL fornecida: 5.4
Registros recebidos: 20
Clientes distintos: 6
Gemini configurado: True
Modelo: gemini-3.5-flash-lite


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


## 2. Diagnóstico e tratamento de qualidade

### Problemas encontrados

1. **ID duplicado:** `OP-0007` aparece duas vezes com conteúdo idêntico.  
   **Tratamento:** manter a primeira ocorrência e remover a cópia para não distorcer volume, mediana, contagens e regras.

2. **Data ausente:** `OP-0017` possui `data = null`.  
   **Tratamento:** manter a operação no histórico e usar `NaT`; ela é excluída apenas da Regra 1, que depende de data. Não há imputação para evitar inventar evidência.

3. **Moedas diferentes:** existem BRL e USD.  
   **Tratamento:** criar `valor_brl` com a taxa fixa fornecida, preservando os campos originais para rastreabilidade.

In [23]:
print("Duplicatas por ID:")
display(df_raw[df_raw.duplicated("id", keep=False)].sort_values("id"))

print("\nValores ausentes por coluna:")
display(df_raw.isna().sum().to_frame("qtd_ausentes"))

print("\nMoedas encontradas:")
display(df_raw["moeda"].value_counts().to_frame("qtd"))

Duplicatas por ID:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,



Valores ausentes por coluna:


,qtd_ausentes
id,0
cliente_id,0
data,1
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



Moedas encontradas:


,qtd
moeda,
BRL,19
USD,1


In [24]:
df = df_raw.drop_duplicates(subset=["id"], keep="first").copy()
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["data_ausente"] = df["data"].isna()

df["valor_brl"] = np.where(
    df["moeda"].str.upper().eq("USD"),
    df["valor"].astype(float) * taxa_cambio,
    df["valor"].astype(float),
)

print(f"Registros após deduplicação: {len(df)}")
print(f"Datas ausentes preservadas: {df['data'].isna().sum()}")

display(df.loc[
    df["moeda"].str.upper().eq("USD"),
    ["id", "cliente_id", "valor", "moeda", "valor_brl"]
])

Registros após deduplicação: 19
Datas ausentes preservadas: 1


,id,cliente_id,valor,moeda,valor_brl
13,OP-0013,CLI-A-4,12000,USD,"64,800.00"


## 3. Agregações pedidas

- volume total transacionado por cliente, em BRL;
- quantidade de operações por canal.

In [25]:
volume_por_cliente = (
    df.groupby("cliente_id", as_index=False)["valor_brl"]
      .sum()
      .rename(columns={"valor_brl": "volume_total_brl"})
      .sort_values("volume_total_brl", ascending=False)
)

operacoes_por_canal = (
    df.groupby("canal", as_index=False)["id"]
      .count()
      .rename(columns={"id": "qtd_operacoes"})
      .sort_values("qtd_operacoes", ascending=False)
)

print("Volume total por cliente:")
display(volume_por_cliente)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Volume total por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,"79,500.00"
0,CLI-A-1,"57,500.00"
1,CLI-A-2,"52,900.00"
2,CLI-A-3,"48,500.00"
4,CLI-A-5,"16,900.00"
5,CLI-A-6,"10,200.00"


Quantidade de operações por canal:


,canal,qtd_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## 4. Regra 1 — Fracionamento

Sinalizar um cliente, em uma mesma data, quando houver **3 ou mais operações**, a soma ultrapassar **R$ 50.000** e nenhuma operação isolada atingir **R$ 20.000**.

In [26]:
resumo_dia = (
    df.dropna(subset=["data"])
      .groupby(["cliente_id", "data"], as_index=False)
      .agg(
          qtd_operacoes_dia=("id", "size"),
          volume_dia_brl=("valor_brl", "sum"),
          maior_operacao_dia_brl=("valor_brl", "max"),
      )
)

resumo_dia["flag_fracionamento_grupo"] = (
    (resumo_dia["qtd_operacoes_dia"] >= 3)
    & (resumo_dia["volume_dia_brl"] > 50_000)
    & (resumo_dia["maior_operacao_dia_brl"] < 20_000)
)

df = df.merge(
    resumo_dia[
        ["cliente_id", "data", "qtd_operacoes_dia", "volume_dia_brl",
         "maior_operacao_dia_brl", "flag_fracionamento_grupo"]
    ],
    on=["cliente_id", "data"],
    how="left",
)
df["flag_fracionamento"] = df["flag_fracionamento_grupo"].fillna(False).astype(bool)

display(resumo_dia[resumo_dia["flag_fracionamento_grupo"]])

/tmp/ipykernel_2245/1105770160.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["flag_fracionamento"] = df["flag_fracionamento_grupo"].fillna(False).astype(bool)


,cliente_id,data,qtd_operacoes_dia,volume_dia_brl,maior_operacao_dia_brl,flag_fracionamento_grupo
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00",True


## 5. Validação explícita da Regra 1

- `CLI-A-1`, 09/03/2026: 3 operações, soma R$ 54.200, maior R$ 18.800 → **capturado**.
- `CLI-A-2`, 14/03/2026: soma R$ 52.900, mas apenas 2 operações e há operação acima de R$ 20 mil → **não capturado**.

In [27]:
casos_validacao = pd.DataFrame({
    "cliente_id": ["CLI-A-1", "CLI-A-2"],
    "data": pd.to_datetime(["2026-03-09", "2026-03-14"]),
    "resultado_esperado": [True, False],
})

validacao_r1 = casos_validacao.merge(
    resumo_dia,
    on=["cliente_id", "data"],
    how="left",
)
validacao_r1["validacao_ok"] = (
    validacao_r1["flag_fracionamento_grupo"]
    == validacao_r1["resultado_esperado"]
)

display(validacao_r1[
    ["cliente_id", "data", "qtd_operacoes_dia", "volume_dia_brl",
     "maior_operacao_dia_brl", "flag_fracionamento_grupo",
     "resultado_esperado", "validacao_ok"]
])

assert validacao_r1["validacao_ok"].all()
print("Validação da Regra 1: OK")

,cliente_id,data,qtd_operacoes_dia,volume_dia_brl,maior_operacao_dia_brl,flag_fracionamento_grupo,resultado_esperado,validacao_ok
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00",True,True,True
1,CLI-A-2,2026-03-14,2,"52,900.00","27,000.00",False,False,True


Validação da Regra 1: OK


## 6. Regra 2 — Valor atípico

Sinalizar uma operação quando o cliente tiver **4 ou mais operações** e `valor_brl > 5 × mediana_cliente_brl`. Todos os cálculos permanecem em pandas.

In [28]:
stats_cliente = (
    df.groupby("cliente_id", as_index=False)
      .agg(
          qtd_operacoes_cliente=("id", "size"),
          mediana_cliente_brl=("valor_brl", "median"),
      )
)

df = df.merge(stats_cliente, on="cliente_id", how="left")
df["limite_atipico_brl"] = 5 * df["mediana_cliente_brl"]
df["flag_valor_atipico"] = (
    (df["qtd_operacoes_cliente"] >= 4)
    & (df["valor_brl"] > df["limite_atipico_brl"])
)

atipicas = df[df["flag_valor_atipico"]].copy()

display(atipicas[
    ["id", "cliente_id", "valor", "moeda", "valor_brl",
     "qtd_operacoes_cliente", "mediana_cliente_brl",
     "limite_atipico_brl", "flag_valor_atipico"]
])

,id,cliente_id,valor,moeda,valor_brl,qtd_operacoes_cliente,mediana_cliente_brl,limite_atipico_brl,flag_valor_atipico
12,OP-0013,CLI-A-4,12000,USD,"64,800.00",4,"5,450.00","27,250.00",True


### Leitura do resultado

`OP-0013`, do `CLI-A-4`, corresponde a US$ 12.000 = **R$ 64.800** pela taxa fixa 5,4. A mediana do cliente é **R$ 5.450** e o limite de 5× é **R$ 27.250**; portanto, a operação é corretamente sinalizada.

## 7. DataFrame final e outputs determinísticos

In [29]:
colunas_exibicao = [
    "id", "cliente_id", "data", "valor", "moeda", "valor_brl",
    "canal", "tipo", "flag_fracionamento", "flag_valor_atipico",
]
display(df[colunas_exibicao])

df_export = df.copy()
df_export["data"] = df_export["data"].dt.strftime("%Y-%m-%d")

df_export.to_csv(OUTPUTS_DIR / "nivel_1_operacoes_tratadas.csv", index=False)
volume_por_cliente.to_csv(OUTPUTS_DIR / "nivel_1_volume_por_cliente.csv", index=False)
operacoes_por_canal.to_csv(OUTPUTS_DIR / "nivel_1_operacoes_por_canal.csv", index=False)
validacao_r1.to_csv(OUTPUTS_DIR / "nivel_1_validacao_fracionamento.csv", index=False)

print("Outputs determinísticos salvos.")

,id,cliente_id,data,valor,moeda,valor_brl,canal,tipo,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,"18,100.00",pix,transferencia_enviada,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,"17,300.00",pix,transferencia_enviada,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,"18,800.00",ted,transferencia_enviada,True,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,"3,300.00",boleto,pagamento,False,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,"25,900.00",ted,transferencia_enviada,False,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,"27,000.00",ted,transferencia_enviada,False,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,"17,200.00",pix,transferencia_enviada,False,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,"15,200.00",pix,transferencia_enviada,False,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,"16,100.00",pix,transferencia_enviada,False,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,"3,800.00",cartao,pagamento,False,False


Outputs determinísticos salvos.


# Parte B — Análise com LLM

O cliente escolhido é **CLI-A-4**, sinalizado pela Regra 2. A LLM recebe somente fatos já calculados em pandas. A saída é estruturada e validada com Pydantic; tokens e latência são registrados.

In [30]:
cliente_escolhido = "CLI-A-4"
caso = df[df["cliente_id"].eq(cliente_escolhido)].copy()

fatos_llm = {
    "cliente_id": cliente_escolhido,
    "qtd_operacoes": int(len(caso)),
    "volume_total_brl": round(float(caso["valor_brl"].sum()), 2),
    "mediana_cliente_brl": round(float(caso["mediana_cliente_brl"].iloc[0]), 2),
    "alertas": {
        "fracionamento": bool(caso["flag_fracionamento"].any()),
        "operacoes_valor_atipico": caso.loc[
            caso["flag_valor_atipico"],
            ["id", "valor_brl", "limite_atipico_brl", "canal", "tipo", "contraparte"]
        ].round({"valor_brl": 2, "limite_atipico_brl": 2}).to_dict(orient="records"),
    },
}

print(json.dumps(fatos_llm, ensure_ascii=False, indent=2, default=str))

{
  "cliente_id": "CLI-A-4",
  "qtd_operacoes": 4,
  "volume_total_brl": 79500.0,
  "mediana_cliente_brl": 5450.0,
  "alertas": {
    "fracionamento": false,
    "operacoes_valor_atipico": [
      {
        "id": "OP-0013",
        "valor_brl": 64800.0,
        "limite_atipico_brl": 27250.0,
        "canal": "ted",
        "tipo": "transferencia_recebida",
        "contraparte": "Zeta Importacao"
      }
    ]
  }
}


## 8. Duas versões de prompt

**V1:** curto e mais aberto.  
**V2:** restringe a análise aos fatos, proíbe recálculo e invenção de contexto e pede uma justificativa auditável.

In [31]:
PROMPT_V1 = f"""
Analise o cliente abaixo sob a ótica de prevenção à lavagem de dinheiro
e produza um parecer estruturado.

Fatos:
{json.dumps(fatos_llm, ensure_ascii=False, default=str)}
""".strip()

PROMPT_V2 = f"""
Atue como analista de Prevenção à Lavagem de Dinheiro.

Regras de trabalho:
1. Use SOMENTE os fatos fornecidos.
2. Não recalcule soma, mediana, contagem ou limites.
3. Não invente contexto cadastral, renda, setor ou intenção.
4. Um alerta determinístico é um indício para triagem, não prova de ilícito.
5. Diferencie evidência observada de interpretação.
6. Se os fatos forem insuficientes para afirmar tipologia específica, declare a limitação.

Fatos determinísticos já calculados por pandas:
{json.dumps(fatos_llm, ensure_ascii=False, default=str)}

Produza um parecer conciso e auditável.
""".strip()

print("PROMPT V1:\n", PROMPT_V1)
print("\n" + "=" * 80 + "\n")
print("PROMPT V2:\n", PROMPT_V2)

PROMPT V1:
 Analise o cliente abaixo sob a ótica de prevenção à lavagem de dinheiro
e produza um parecer estruturado.

Fatos:
{"cliente_id": "CLI-A-4", "qtd_operacoes": 4, "volume_total_brl": 79500.0, "mediana_cliente_brl": 5450.0, "alertas": {"fracionamento": false, "operacoes_valor_atipico": [{"id": "OP-0013", "valor_brl": 64800.0, "limite_atipico_brl": 27250.0, "canal": "ted", "tipo": "transferencia_recebida", "contraparte": "Zeta Importacao"}]}}


PROMPT V2:
 Atue como analista de Prevenção à Lavagem de Dinheiro.

Regras de trabalho:
1. Use SOMENTE os fatos fornecidos.
2. Não recalcule soma, mediana, contagem ou limites.
3. Não invente contexto cadastral, renda, setor ou intenção.
4. Um alerta determinístico é um indício para triagem, não prova de ilícito.
5. Diferencie evidência observada de interpretação.
6. Se os fatos forem insuficientes para afirmar tipologia específica, declare a limitação.

Fatos determinísticos já calculados por pandas:
{"cliente_id": "CLI-A-4", "qtd_operac

In [32]:
from typing import Literal
from pydantic import BaseModel, ValidationError
from google import genai
from google.genai import types

class ParecerPLD(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

def chamar_gemini(prompt: str, max_retries: int = 1):
    """Chamada estruturada com validação, retry, latência e tokens."""
    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY não configurada.")

    model = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
    client = genai.Client(api_key=api_key)
    erro_anterior = None

    for tentativa in range(max_retries + 1):
        prompt_atual = prompt
        if tentativa > 0:
            prompt_atual += (
                "\n\nA resposta anterior não validou. "
                "Retorne somente uma resposta compatível com o schema exigido."
            )

        t0 = time.perf_counter()
        response = client.models.generate_content(
            model=model,
            contents=prompt_atual,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ParecerPLD,
                temperature=0.2,
            ),
        )
        latencia = time.perf_counter() - t0

        try:
            parecer = ParecerPLD.model_validate_json(response.text)
            usage = getattr(response, "usage_metadata", None)
            metricas = {
                "modelo": model,
                "latencia_s": round(latencia, 4),
                "tokens_entrada": getattr(usage, "prompt_token_count", None),
                "tokens_saida": getattr(usage, "candidates_token_count", None),
                "tokens_total": getattr(usage, "total_token_count", None),
                "tentativa": tentativa + 1,
            }
            return parecer, metricas
        except ValidationError as exc:
            erro_anterior = exc

    raise RuntimeError(f"Resposta malformada após retry: {erro_anterior}")

print("Schema e função estruturada prontos.")

Schema e função estruturada prontos.


In [ ]:
resultados_prompts = {}

if not os.getenv("GEMINI_API_KEY", "").strip():
    print("LLM não executada: configure GEMINI_API_KEY e execute novamente.")
else:
    for nome, prompt in {"v1": PROMPT_V1, "v2": PROMPT_V2}.items():
        parecer, metricas = chamar_gemini(prompt)
        resultados_prompts[nome] = {
            "parecer": parecer.model_dump(),
            "metricas": metricas,
        }
        print(f"\n{nome.upper()}")
        print(json.dumps(resultados_prompts[nome], ensure_ascii=False, indent=2))

    with open(
        OUTPUTS_DIR / "nivel_1_comparacao_prompts.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(resultados_prompts, f, ensure_ascii=False, indent=2)


V1
{
  "parecer": {
    "nivel_risco": "médio",
    "tipologia_suspeita": "Transação atípica de alto valor sem justificativa aparente",
    "red_flags": [
      "Operação de valor atípico significativamente superior à mediana histórica do cliente",
      "Recebimento de recursos via TED de pessoa jurídica (Zeta Importacao)"
    ],
    "justificativa": "O cliente apresentou um volume total movimentado de R$ 79.500,00 em poucas operações, impulsionado principalmente por uma única transferência recebida de R$ 64.800,00 da empresa Zeta Importacao, valor este que supera em mais de duas vezes o limite considerado atípico para o seu perfil (R$ 27.250,00) e fica muito acima de sua mediana de R$ 5.450,00. Embora o fracionamento tenha sido descartado, a ocorrência de uma operação com contraparte empresarial de grande monta destoa do comportamento padrão, justificando o monitoramento e a averiguação da origem dos recursos."
  },
  "metricas": {
    "modelo": "gemini-3.5-flash-lite",
    "latenci

In [ ]:
if resultados_prompts:
    comparacao_prompts = pd.DataFrame([
        {
            "versao": nome.upper(),
            "nivel_risco": r["parecer"]["nivel_risco"],
            "tipologia_suspeita": r["parecer"]["tipologia_suspeita"],
            "qtd_red_flags": len(r["parecer"]["red_flags"]),
            "latencia_s": r["metricas"]["latencia_s"],
            "tokens_entrada": r["metricas"]["tokens_entrada"],
            "tokens_saida": r["metricas"]["tokens_saida"],
            "tokens_total": r["metricas"].get("tokens_total"),
        }
        for nome, r in resultados_prompts.items()
    ])
    display(comparacao_prompts)
else:
    print("Execute primeiro as chamadas V1 e V2.")

### Comparação dos prompts

Na execução observada, as duas versões chegaram ao mesmo nível geral de risco, mas o **V2 foi mais conservador e auditável** na tipologia. O V1 chegou a sugerir incompatibilidade com um “perfil financeiro” que não foi fornecido explicitamente; o V2 ficou mais restrito à atipicidade presente nos fatos.

O V2 possui mais instruções e tende a consumir mais tokens de entrada. Esse custo adicional é aceito como trade-off por maior controle sobre extrapolações e melhor separação entre cálculo determinístico e interpretação. A tabela acima registra os valores exatos de tokens e latência de cada execução.

## 9. Tratamento de resposta malformada

A resposta é validada novamente com Pydantic antes de ser usada. O teste abaixo mostra que valores fora do schema, tipos incorretos e campos ausentes são rejeitados de forma controlada.

In [ ]:
def validar_resposta_llm(texto_resposta: str):
    try:
        parecer = ParecerPLD.model_validate_json(texto_resposta)
        return {"valido": True, "parecer": parecer.model_dump(), "erro": None}
    except ValidationError as exc:
        return {"valido": False, "parecer": None, "erro": str(exc)}
    except Exception as exc:
        return {"valido": False, "parecer": None, "erro": f"Erro inesperado: {exc}"}

resposta_malformada_teste = """
{
  "nivel_risco": "critico",
  "tipologia_suspeita": "teste",
  "red_flags": "deveria ser uma lista"
}
"""

teste_validacao = validar_resposta_llm(resposta_malformada_teste)
print("Resposta válida:", teste_validacao["valido"])
print("Erro capturado:")
print(teste_validacao["erro"])

## Conclusão do Nível 1

A base foi limpa sem imputar evidências inexistentes, os valores foram normalizados para BRL e as duas regras foram calculadas exclusivamente em pandas. A Regra 1 foi validada com um caso positivo e um negativo, e a Regra 2 identificou a operação atípica esperada.

Na etapa de LLM, o modelo recebeu apenas fatos determinísticos já calculados. A saída foi estruturada e validada, com medição de tokens e latência, comparação entre duas versões de prompt e tratamento explícito de resposta malformada.